In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('train.csv')

#separate features and make 'price' target
X = df.drop(columns=['price'])
y = df['price']

#scale features so mean = 0 and st dev = 1 w/ StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

#dvide the price by 1000
y_scaled = y / 1000

print("First 5 rows of scaled features:")
print(X_scaled_df.head())
print("\nFirst 5 rows of scaled target (price):")
print(y_scaled.head())

First 5 rows of scaled features:
   Unnamed: 0  bedrooms  bathrooms  sqft_living  sqft_lot    floors  \
0   -1.730320 -0.409823  -1.449888    -0.981646 -0.312717 -0.863477   
1   -1.726856 -0.409823   0.283184     0.584578 -0.257719  1.070402   
2   -1.723391 -1.584103  -1.449888    -1.443625 -0.162440 -0.863477   
3   -1.719927  0.764456   1.323028    -0.102758 -0.335172 -0.863477   
4   -1.716463 -0.409823  -0.063430    -0.418256 -0.228769 -0.863477   

   waterfront      view  condition     grade  sqft_above  sqft_basement  \
0   -0.089803 -0.309908  -0.673452 -0.522576   -0.722231      -0.667586   
1   -0.089803 -0.309908  -0.673452 -0.522576    0.531438       0.219976   
2   -0.089803 -0.309908  -0.673452 -1.384913   -1.241427      -0.667586   
3   -0.089803 -0.309908   2.229358 -0.522576   -0.886854       1.351617   
4   -0.089803 -0.309908  -0.673452  0.339761   -0.089065      -0.667586   

   yr_built  yr_renovated   zipcode       lat      long  sqft_living15  \
0 -0.498602    

Problem 2




In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')


drop_cols = ['id', 'date', 'zipcode']
drop_cols2 = ['zipcode']
train_df = train_df.drop(columns=drop_cols2)
test_df = test_df.drop(columns=drop_cols)

#separate X and y
X_train = train_df.drop(columns=['price'])
y_train = train_df['price']
X_test = test_df.drop(columns=['price'])
y_test = test_df['price']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_scaled = y_train / 1000
y_test_scaled = y_test / 1000


#train model
model = LinearRegression()
model.fit(X_train_scaled, y_train_scaled)

#get preds
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

#calc mse and r2
train_mse = mean_squared_error(y_train_scaled, y_pred_train)
train_r2 = r2_score(y_train_scaled, y_pred_train)

test_mse = mean_squared_error(y_test_scaled, y_pred_test)
test_r2 = r2_score(y_test_scaled, y_pred_test)

print('Problem 2 Results:')
print("Training Metrics")
print(f"MSE: {train_mse:.4f}")
print(f"R2:  {train_r2:.4f}")

print("\nTesting Metrics")
print(f"MSE: {test_mse:.4f}")
print(f"R2:  {test_r2:.4f}")

print("\nFeature Coefficients")
#create table
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
})
#sort by absolute value
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
print(coef_df.sort_values(by='Abs_Coefficient', ascending=False))

Problem 2 Results:
Training Metrics
MSE: 31415.7479
R2:  0.7271

Testing Metrics
MSE: 58834.6740
R2:  0.6471

Feature Coefficients
          Feature  Coefficient  Abs_Coefficient
9           grade    92.511076        92.511076
14            lat    78.129852        78.129852
12       yr_built   -68.043173        68.043173
6      waterfront    64.230911        64.230911
3     sqft_living    57.161582        57.161582
10     sqft_above    48.439051        48.439051
7            view    47.610288        47.610288
16  sqft_living15    45.479128        45.479128
11  sqft_basement    27.688812        27.688812
2       bathrooms    18.456913        18.456913
13   yr_renovated    17.341926        17.341926
17     sqft_lot15   -12.906560        12.906560
1        bedrooms   -12.807339        12.807339
8       condition    12.647609        12.647609
4        sqft_lot    11.127338        11.127338
0      Unnamed: 0     8.456024         8.456024
5          floors     8.151038         8.151038
15   

Problem 3:

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

drop_cols = ['id', 'date', 'zipcode']
drop_cols2 = ['zipcode']
train_df = train_df.drop(columns=drop_cols2)
test_df = test_df.drop(columns=drop_cols)


X_train_raw = train_df.drop(columns=['price'])
y_train_raw = train_df['price']
X_test_raw = test_df.drop(columns=['price'])
y_test_raw = test_df['price']

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

y_train = y_train_raw.values / 1000
y_test = y_test_raw.values / 1000


def train_closed_form(X, y):
    """
    function computes the weights using the Normal Equation:
    theta = (X^T * X)^-1 * X^T * y
    """
    m = X.shape[0]
    ones = np.ones((m, 1))
    X_b = np.hstack((ones, X))

    #normal equation
    X_T = X_b.T
    theta = np.linalg.inv(X_T @ X_b) @ X_T @ y

    return theta

def predict_closed_form(X, theta):
    """
    this function predicts the response variable y_pred = X * theta
    """
    m = X.shape[0]
    ones = np.ones((m, 1))
    X_b = np.hstack((ones, X))

    return X_b @ theta

theta_my = train_closed_form(X_train, y_train)

y_pred_train_my = predict_closed_form(X_train, theta_my)
y_pred_test_my = predict_closed_form(X_test, theta_my)

mse_train_my = mean_squared_error(y_train, y_pred_train_my)
r2_train_my = r2_score(y_train, y_pred_train_my)
mse_test_my = mean_squared_error(y_test, y_pred_test_my)
r2_test_my = r2_score(y_test, y_pred_test_my)

sk_model = LinearRegression()
sk_model.fit(X_train, y_train)
y_pred_test_sk = sk_model.predict(X_test)
mse_test_sk = mean_squared_error(y_test, y_pred_test_sk)


print('Problem 3 Results:')
print("My Closed-Form Implementation Results")
print(f"Training MSE: {mse_train_my:.4f}")
print(f"Training R2:  {r2_train_my:.4f}")
print(f"Testing MSE:  {mse_test_my:.4f}")
print(f"Testing R2:   {r2_test_my:.4f}")

print("\nComparison Check")
print(f"My Implementation MSE:      {mse_test_my:.4f}")
print(f"Scikit-Learn Package MSE:   {mse_test_sk:.4f}")

if np.isclose(mse_test_my, mse_test_sk):
    print("\nCONCLUSION: The results are the same.")
else:
    print("\nCONCLUSION: The results are different")

Problem 3 Results:
My Closed-Form Implementation Results
Training MSE: 32038.6927
Training R2:  0.7217
Testing MSE:  60800.9275
Testing R2:   0.6353

Comparison Check
My Implementation MSE:      60800.9275
Scikit-Learn Package MSE:   58834.6740

CONCLUSION: The results are different


Problem 4:


In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# using [[ ]] to keep df 2d
X_train_raw = train_df[['sqft_living']].values
y_train = train_df['price'].values / 1000

X_test_raw = test_df[['sqft_living']].values
y_test = test_df['price'].values / 1000


def create_polynomial_features(X, degree):
    """
    function creates polynomial features up to the given degree.
    """
    X_poly = X.copy()
    for p in range(2, degree + 1):
        new_col = np.power(X, p)
        X_poly = np.hstack((X_poly, new_col))
    return X_poly

def train_closed_form(X, y):
    """
    my implementation from Problem 3
    """
    m = X.shape[0]
    ones = np.ones((m, 1))
    X_b = np.hstack((ones, X))

    # Calculate Theta
    X_T = X_b.T
    # Using pinv for stability with polynomial features
    theta = np.linalg.pinv(X_T @ X_b) @ X_T @ y
    return theta

def predict_closed_form(X, theta):
    m = X.shape[0]
    ones = np.ones((m, 1))
    X_b = np.hstack((ones, X))
    return X_b @ theta


results = []

print(f"{'Degree':<8} | {'Train MSE':<12} | {'Train R2':<10} | {'Test MSE':<12} | {'Test R2':<10}")
print("-" * 65)

degrees_to_test = [1, 2, 3, 4, 5]

for p in degrees_to_test:
    X_train_poly = create_polynomial_features(X_train_raw, p)
    X_test_poly = create_polynomial_features(X_test_raw, p)

    #scaling data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_poly)
    X_test_scaled = scaler.transform(X_test_poly)

    #training Model
    theta = train_closed_form(X_train_scaled, y_train)

    #evaluate
    #predict on train
    y_train_pred = predict_closed_form(X_train_scaled, theta)
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    #predict on Test
    y_test_pred = predict_closed_form(X_test_scaled, theta)
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    #print Row
    print(f"{p:<8} | {train_mse:<12.4f} | {train_r2:<10.4f} | {test_mse:<12.4f} | {test_r2:<10.4f}")

Degree   | Train MSE    | Train R2   | Test MSE     | Test R2   
-----------------------------------------------------------------
1        | 57947.5262   | 0.4967     | 88575.9785   | 0.4687    
2        | 54822.6651   | 0.5238     | 71791.6795   | 0.5694    
3        | 53785.1947   | 0.5329     | 99833.4838   | 0.4012    
4        | 52795.7748   | 0.5415     | 250979.2743  | -0.5053   
5        | 52626.1120   | 0.5429     | 28657284.7532 | -170.8815 


Problem 5

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score


train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

drop_cols = ['id', 'date', 'zipcode']
drop_cols2 = ['zipcode']
train_df = train_df.drop(columns=drop_cols2)
test_df = test_df.drop(columns=drop_cols)

X_train_raw = train_df.drop(columns=['price'])
y_train_raw = train_df['price']
X_test_raw = test_df.drop(columns=['price'])
y_test_raw = test_df['price']


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)


y_train = y_train_raw.values / 1000
y_test = y_test_raw.values / 1000


m_train = X_train.shape[0]
X_train_b = np.hstack((np.ones((m_train, 1)), X_train))

m_test = X_test.shape[0]
X_test_b = np.hstack((np.ones((m_test, 1)), X_test))



def train_gradient_descent(X, y, learning_rate, iterations):
    """
    Trains Linear Regression using Gradient Descent.
    X: Feature matrix with bias term added (N x D+1)
    y: Target vector (N,)
    """
    m, n = X.shape
    theta = np.zeros(n)


    for i in range(iterations):
        #calc preds
        predictions = X.dot(theta)

        #calc error
        errors = predictions - y

        #calc gradient
        gradient = (2/m) * X.T.dot(errors)

        #update weights
        theta = theta - (learning_rate * gradient)

    return theta

learning_rates = [0.01, 0.1, 0.5]
iteration_counts = [10, 50, 100]

print(f"{'Alpha':<6} | {'Iters':<5} | {'Train MSE':<12} | {'Train R2':<10} | {'Test MSE':<12} | {'Test R2':<10}")
print("-" * 70)

for alpha in learning_rates:
    for iters in iteration_counts:
        theta = train_gradient_descent(X_train_b, y_train, alpha, iters)

        y_pred_train = X_train_b.dot(theta)
        y_pred_test = X_test_b.dot(theta)

        mse_train = mean_squared_error(y_train, y_pred_train)
        r2_train = r2_score(y_train, y_pred_train)
        mse_test = mean_squared_error(y_test, y_pred_test)
        r2_test = r2_score(y_test, y_pred_test)

        print(f"{alpha:<6} | {iters:<5} | {mse_train:<12.4f} | {r2_train:<10.4f} | {mse_test:<12.4f} | {r2_test:<10.4f}")

Alpha  | Iters | Train MSE    | Train R2   | Test MSE     | Test R2   
----------------------------------------------------------------------
0.01   | 10    | 235731.0662  | -1.0474    | 282866.8035  | -0.6966   
0.01   | 50    | 69695.7840   | 0.3947     | 94320.0251   | 0.4343    
0.01   | 100   | 36764.9459   | 0.6807     | 61279.0373   | 0.6325    
0.1    | 10    | 35047.9288   | 0.6956     | 60003.7888   | 0.6401    
0.1    | 50    | 31427.0633   | 0.7270     | 58890.5372   | 0.6468    
0.1    | 100   | 31416.0180   | 0.7271     | 58839.9318   | 0.6471    
0.5    | 10    | 146443379626902912.0000 | -1271903546248.4849 | 163245223878896064.0000 | -979117206654.7434
0.5    | 50    | 12938672170858728061778061232738426634003445727110124147184058236928.0000 | -112376148787346696548726695715411919899851430096269827071016960.0000 | 14423159203771321688028602233766020754586700553518676346734787952640.0000 | -86507666289974097354925205133071170505605562858981695240011776.0000
0.5    | 100

Problem 6:


In [7]:
#P6 Part 2

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

def train_ridge_gradient_descent(X, y, learning_rate, iterations, lambda_val):
    """
    Trains Ridge Regression using Gradient Descent.
    Loss = SSE + lambda * sum(theta_j^2)
    """
    m, n = X.shape
    theta = np.zeros(n)

    for i in range(iterations):

        predictions = X.dot(theta)


        errors = predictions - y

        #calc gradient (SSE: 2 * X.T * errors)
        grad_sse = 2 * X.T.dot(errors)

        # Gradient regularization termc(2 * lambda * theta)
        theta_reg = theta.copy()
        theta_reg[0] = 0
        grad_reg = 2 * lambda_val * theta_reg


        gradient = grad_sse + grad_reg

        #update weights
        theta = theta - (learning_rate / m) * gradient

    return theta

In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score


np.random.seed(42)
N = 1000

X = np.random.uniform(-2, 2, (N, 1))
e = np.random.normal(0, np.sqrt(2), (N, 1))
y = 1 + 2*X + e


lambdas = [0, 1, 10, 100, 1000, 10000]

results = []

print(f"{'Lambda':<8} | {'Slope':<10} | {'MSE':<12} | {'R2 Score':<10}")
print("-" * 50)

for lam in lambdas:
    if lam == 0:
        #standard LR
        model = LinearRegression()
    else:
        #ridge reg
        #sklearn uses alpha for lambda
        model = Ridge(alpha=lam)

    model.fit(X, y)
    y_pred = model.predict(X)

    #extracting the scalar slope
    slope = model.coef_.item()
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    print(f"{lam:<8} | {slope:<10.4f} | {mse:<12.4f} | {r2:<10.4f}")

Lambda   | Slope      | MSE          | R2 Score  
--------------------------------------------------
0        | 1.9453     | 1.9499       | 0.7258    
1        | 1.9439     | 1.9499       | 0.7258    
10       | 1.9311     | 1.9502       | 0.7258    
100      | 1.8124     | 1.9740       | 0.7224    
1000     | 1.1225     | 2.8735       | 0.5960    
10000    | 0.2335     | 5.9471       | 0.1638    
